# AgriSense AI: Crop Price Forecasting, Climate Simulation & Smart Recommendations 🌾📈

Welcome to **AgriSense AI**, an interactive, data-driven intelligence dashboard built to help farmers, agricultural traders, and analysts predict crop prices, evaluate climate risks, identify price crashes, and find the best markets to sell crop loads.

### Core Features:
1. **Interactive Price Forecasting**: Look up any of the 80 APMC markets and view weekly historical price/arrival trends alongside 4-week forecasts.
2. **Climate Impact Simulator**: Simulate weather anomalies (e.g. droughts, temperature hikes) and see how they impact predicted crop prices.
3. **Best Market Recommendation**: Identify the top-performing markets in the region to sell your crop loads to maximize profit.
4. **Price Crash Alerts**: Get warning flags for markets where prices are predicted to drop significantly in the upcoming weeks.
5. **AI Farmer Chatbot**: Chat with an AI assistant that answers queries about forecasts, crash warnings, and optimal selling locations.

---
### Instructions to Run:
1. Make sure `final_dataset_ready.csv` is uploaded to the same directory as this notebook (or in your Colab workspace).
2. Click **Runtime > Run All** (or press `Ctrl+F9`) to train the machine learning model and launch the interactive dashboard below!

In [ ]:
# Cell 1: Install and import dependencies
!pip install -q ipywidgets plotly scikit-learn pandas numpy google-generativeai

import pandas as pd
import numpy as np
import plotly.graph_objects as go
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
import warnings
warnings.filterwarnings('ignore')
print("Setup Complete! Dependencies imported.")

In [ ]:
# Cell 2: Load and preprocess the dataset
import os
csv_path = "final_dataset_ready.csv"

if not os.path.exists(csv_path):
    raise FileNotFoundError(
        f"Could not find '{csv_path}'. Please upload final_dataset_ready.csv to the Colab files tab."
    )

df = pd.read_csv(csv_path)
print(f"Dataset loaded successfully! Shape: {df.shape}")

# Clean missing values per market using forward-fill and backward-fill
print("Cleaning missing lag features market-by-market...")
for market in df['Market'].unique():
    mask = df['Market'] == market
    df.loc[mask] = df.loc[mask].ffill().bfill()

# Fallback fill for any remaining NaNs
still_null = df.isnull().sum().sum()
if still_null > 0:
    df = df.fillna(df.mean(numeric_only=True))

print(f"Data cleaning complete. 0 null values remaining. Total unique markets: {df['Market'].nunique()}")

In [ ]:
# Cell 3: Train the Machine Learning Model
# Selecting features for forecasting (excluding contemporaneous leakage features like neigh_price/neigh_price_diff)
features = [
    'Market_id', 'Month_Num', 'Week_Of_Month', 'time_index', 'month_sin', 'month_cos',
    'Rainfall', 'rain_lag_1', 'rain_lag_2', 'rain_lag_4', 'rain_lag_8', 
    'rain_cum_4w', 'rain_cum_8w', 'rain_cum_12w', 'is_monsoon', 'is_dry', 
    'monsoon_total', 'monsoon_prev_year', 'temp_mean', 'temp_min', 'temp_max', 'temp_range', 
    'temp_mean_lag_1', 'temp_mean_lag_2', 'temp_mean_lag_3', 'temp_mean_lag_4', 
    'temp_range_lag_1', 'temp_range_lag_2', 'temp_mean_roll_2', 'temp_mean_roll_4', 
    'has_neighbors', 
    'neigh_price_lag_1', 'neigh_price_lag_2', 'neigh_price_lag_4', 
    'neigh_price_diff_lag_1', 'neigh_price_diff_lag_2', 'neigh_price_diff_lag_4'
]

print("Training Predictive Random Forest Regressor (50 estimators) on all historical data...")
rf_model = RandomForestRegressor(n_estimators=50, random_state=42, n_jobs=-1)
rf_model.fit(df[features], df['Price_Rs_Per_Quintal'])

# Check fit score
fit_score = r2_score(df['Price_Rs_Per_Quintal'], rf_model.predict(df[features]))
print(f"Training complete! Model Fit R2 Score: {fit_score:.4f}")

In [ ]:
# Cell 4: Define Dashboard Calculation Logic

def generate_forecast(market_name):
    """Generates 4 weeks of baseline forecast (weeks 289 to 292) for a selected market"""
    market_df = df[df['Market'] == market_name]
    if market_df.empty:
        return pd.DataFrame()
    
    market_id = market_df['Market_id'].iloc[0]
    future_weeks = [289, 290, 291, 292]
    future_rows = []
    
    for t in future_weeks:
        year = 2020 + (t - 1) // 48
        week_in_year = (t - 1) % 48
        month = week_in_year // 4 + 1
        week = week_in_year % 4 + 1
        
        # Use average historical variables for the same calendar month and week
        hist_week = market_df[(market_df['Month_Num'] == month) & (market_df['Week_Of_Month'] == week)]
        if hist_week.empty:
            hist_week = market_df[market_df['Month_Num'] == month]
            
        mean_features = hist_week[features].mean()
        mean_features['Year'] = year
        mean_features['time_index'] = t
        mean_features['Market_id'] = market_id
        mean_features['Month_Num'] = month
        mean_features['Week_Of_Month'] = week
        
        future_rows.append(mean_features)
        
    forecast_df = pd.DataFrame(future_rows)
    forecast_df['Predicted_Price'] = rf_model.predict(forecast_df[features])
    return forecast_df

def generate_climate_forecast(market_name, rain_change_pct, temp_change_val):
    """Generates 4 weeks of future forecasts under simulated climate conditions"""
    forecast_df = generate_forecast(market_name)
    if forecast_df.empty:
        return pd.DataFrame()
        
    sim_df = forecast_df.copy()
    
    # Scale rainfall variables
    rain_cols = ['Rainfall', 'rain_lag_1', 'rain_lag_2', 'rain_lag_4', 'rain_lag_8', 
                 'rain_cum_4w', 'rain_cum_8w', 'rain_cum_12w', 'monsoon_total', 'monsoon_prev_year']
    for col in rain_cols:
        sim_df[col] = sim_df[col] * (1.0 + rain_change_pct / 100.0)
        
    # Shift temperature variables
    temp_cols = ['temp_mean', 'temp_min', 'temp_max', 'temp_mean_lag_1', 'temp_mean_lag_2', 
                 'temp_mean_lag_3', 'temp_mean_lag_4', 'temp_mean_roll_2', 'temp_mean_roll_4']
    for col in temp_cols:
        sim_df[col] = sim_df[col] + temp_change_val
        
    sim_df['Predicted_Price'] = rf_model.predict(sim_df[features])
    return sim_df

def get_best_markets(target_week=289):
    """Finds top 5 markets with the highest predicted prices for a future week"""
    results = []
    for m in df['Market'].unique():
        f_df = generate_forecast(m)
        row = f_df[f_df['time_index'] == target_week]
        if not row.empty:
            results.append({
                'Market': m,
                'District': df[df['Market'] == m]['District'].iloc[0],
                'Predicted_Price': row['Predicted_Price'].iloc[0]
            })
    return pd.DataFrame(results).sort_values('Predicted_Price', ascending=False)

def get_price_crash_alerts(target_week=289):
    """Finds markets where predicted prices drop by >10% compared to their week 288 historical price"""
    alerts = []
    for m in df['Market'].unique():
        hist_last = df[(df['Market'] == m) & (df['time_index'] == 288)]['Price_Rs_Per_Quintal'].iloc[0]
        f_df = generate_forecast(m)
        row = f_df[f_df['time_index'] == target_week]
        if not row.empty:
            pred_val = row['Predicted_Price'].iloc[0]
            drop_pct = ((hist_last - pred_val) / hist_last) * 100
            if drop_pct > 10.0: # Drop is positive when price falls
                alerts.append({
                    'Market': m,
                    'District': df[df['Market'] == m]['District'].iloc[0],
                    'Last_Price': hist_last,
                    'Predicted_Price': pred_val,
                    'Drop_Pct': drop_pct
                })
    if not alerts:
        return pd.DataFrame(columns=['Market', 'District', 'Last_Price', 'Predicted_Price', 'Drop_Pct'])
    return pd.DataFrame(alerts).sort_values('Drop_Pct', ascending=False)

print("Helper logic defined!")

In [ ]:
# Cell 5: Create and Style Dashboard Layout

# Custom CSS injection for premium styling
style_html = """
<style>
    .dashboard-title-card {
        background: linear-gradient(135deg, #0e2b1d, #06140e);
        border-left: 6px solid #2ec4b6;
        padding: 20px;
        border-radius: 8px;
        margin-bottom: 20px;
        color: #ffffff;
    }
    .dashboard-title-card h1 {
        margin: 0 0 5px 0;
        font-size: 2.2em;
        font-family: 'Outfit', sans-serif;
        letter-spacing: 0.5px;
    }
    .dashboard-title-card p {
        margin: 0;
        color: #a3c2b2;
        font-size: 1.1em;
    }
    .card-grid {
        display: grid;
        grid-template-columns: repeat(auto-fit, minmax(220px, 1fr));
        gap: 15px;
        margin-bottom: 20px;
    }
    .stat-card {
        background: #111e19;
        border: 1px solid #1a3327;
        border-radius: 8px;
        padding: 15px;
        color: #e0e0e0;
        box-shadow: 0 4px 6px rgba(0,0,0,0.2);
    }
    .stat-card h3 {
        margin: 0 0 8px 0;
        font-size: 0.9em;
        color: #8bada0;
        text-transform: uppercase;
        letter-spacing: 0.5px;
    }
    .stat-card .val {
        font-size: 1.8em;
        font-weight: 700;
        color: #ffffff;
        margin-bottom: 3px;
    }
    .stat-card .desc {
        font-size: 0.85em;
        color: #a3c2b2;
    }
    .alert-card-grid {
        display: grid;
        grid-template-columns: repeat(auto-fill, minmax(280px, 1fr));
        gap: 12px;
        margin-top: 10px;
    }
    .alert-card {
        background: linear-gradient(135deg, #2b1717, #1c0e0e);
        border: 1px solid #6b2828;
        border-radius: 8px;
        padding: 15px;
        color: #ffe0e0;
        box-shadow: 0 4px 8px rgba(0,0,0,0.3);
    }
    .alert-header {
        font-weight: bold;
        font-size: 1.15em;
        color: #ff6b6b;
        margin-bottom: 6px;
    }
    .recommend-card-grid {
        display: flex;
        flex-direction: column;
        gap: 10px;
        margin-top: 10px;
    }
    .recommend-card {
        background: linear-gradient(135deg, #13271d, #0b1a13);
        border: 1px solid #1c4d37;
        border-radius: 8px;
        padding: 15px;
        color: #e0ffe8;
        box-shadow: 0 4px 8px rgba(0,0,0,0.3);
    }
    .recommend-header {
        font-weight: bold;
        font-size: 1.15em;
        color: #2ec4b6;
        margin-bottom: 6px;
    }
    .chat-container {
        height: 380px;
        overflow-y: auto;
        border: 1px solid #1c3d2f;
        background-color: #08100d;
        padding: 15px;
        border-radius: 8px;
        margin-bottom: 12px;
        display: flex;
        flex-direction: column;
        gap: 12px;
    }
    .chat-bubble-user {
        background-color: #1a4d34;
        color: #ffffff;
        padding: 10px 14px;
        border-radius: 16px 16px 0 16px;
        max-width: 75%;
        align-self: flex-end;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    }
    .chat-bubble-bot {
        background-color: #16241f;
        color: #e5f5ed;
        padding: 10px 14px;
        border-radius: 16px 16px 16px 0;
        max-width: 75%;
        align-self: flex-start;
        border-left: 4px solid #2ec4b6;
        box-shadow: 0 2px 4px rgba(0,0,0,0.2);
    }
    .quick-chips-container {
        display: flex;
        gap: 8px;
        flex-wrap: wrap;
        margin-bottom: 10px;
    }
    .quick-chip-btn {
        background-color: #142b21;
        border: 1px solid #1c3d2f;
        color: #a3c2b2;
        padding: 6px 12px;
        border-radius: 16px;
        font-size: 0.85em;
        cursor: pointer;
        transition: all 0.2s;
    }
    .quick-chip-btn:hover {
        background-color: #1a4d34;
        color: #ffffff;
        border-color: #2ec4b6;
    }
</style>
"""
display(HTML(style_html))

# Define interactive widgets
market_dropdown = widgets.Dropdown(
    options=sorted(df['Market'].unique()),
    value="Ahmednagar APMC",
    description="Select Market:",
    style={'description_width': 'initial'}
)

rain_slider = widgets.FloatSlider(
    value=0.0,
    min=-100.0,
    max=100.0,
    step=10.0,
    description="Rainfall Deficit/Surplus (%):",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

temp_slider = widgets.FloatSlider(
    value=0.0,
    min=-5.0,
    max=5.0,
    step=0.5,
    description="Temperature Shift (°C):",
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='80%')
)

sim_btn = widgets.Button(
    description="Run Climate Simulation",
    button_style='success',
    icon='bolt'
)

reset_sim_btn = widgets.Button(
    description="Reset Sliders",
    button_style='warning',
    icon='undo'
)

# Output widgets
output_forecast = widgets.Output()
output_recommend = widgets.Output()
output_simulator = widgets.Output()
output_alerts = widgets.Output()
output_chat = widgets.Output()
output_summary_cards = widgets.Output()

print("Dashboard UI layout and CSS loaded!")

In [ ]:
# Cell 6: Render Key Summary Metrics Cards

def render_summary_cards():
    with output_summary_cards:
        clear_output()
        # Calculations
        avg_price = df[df['time_index'] == 288]['Price_Rs_Per_Quintal'].mean()
        top_market_row = df[df['time_index'] == 288].sort_values('Price_Rs_Per_Quintal', ascending=False).iloc[0]
        crashes_df = get_price_crash_alerts(target_week=289)
        crash_count = len(crashes_df)
        
        html_content = f"""
        <div class="card-grid">
            <div class="stat-card">
                <h3>Statewide Avg Price</h3>
                <div class="val">Rs {avg_price:.2f}</div>
                <div class="desc">Average across all 80 markets (Week 288)</div>
            </div>
            <div class="stat-card">
                <h3>Premium Selling Market</h3>
                <div class="val" style="font-size: 1.4em; margin: 4px 0;">{top_market_row['Market'].replace(' APMC', '')}</div>
                <div class="desc">Highest Price: Rs {top_market_row['Price_Rs_Per_Quintal']:.2f}/q</div>
            </div>
            <div class="stat-card" style="border-color: { '#1a3327' if crash_count == 0 else '#6b2828' };">
                <h3>Price Crash Warnings</h3>
                <div class="val" style="color: { '#ffffff' if crash_count == 0 else '#ff6b6b' };">{crash_count} Markets</div>
                <div class="desc">Markets with predicted drops >10% next week</div>
            </div>
        </div>
        """
        display(HTML(html_content))

print("Summary cards renderer defined!")

In [ ]:
# Cell 7: Render Interactive Charts & Recommendations

def update_forecast_tab(change=None):
    market_name = market_dropdown.value
    
    # Update summary cards dynamically
    render_summary_cards()
    
    # Generate Forecast
    forecast_df = generate_forecast(market_name)
    if forecast_df.empty:
        return
        
    # Get last 12 historical records
    market_df = df[df['Market'] == market_name]
    hist_recent = market_df.tail(12)
    
    with output_forecast:
        clear_output(wait=True)
        
        fig = go.Figure()
        
        # Historical line
        fig.add_trace(go.Scatter(
            x=hist_recent['time_index'],
            y=hist_recent['Price_Rs_Per_Quintal'],
            name="Historical Price",
            line=dict(color='#2ec4b6', width=3.5),
            mode='lines+markers',
            marker=dict(size=8)
        ))
        
        # Connecting point (Week 288)
        connecting_x = [hist_recent['time_index'].iloc[-1], forecast_df['time_index'].iloc[0]]
        connecting_y = [hist_recent['Price_Rs_Per_Quintal'].iloc[-1], forecast_df['Predicted_Price'].iloc[0]]
        
        fig.add_trace(go.Scatter(
            x=connecting_x,
            y=connecting_y,
            showlegend=False,
            line=dict(color='#ffb703', width=3, dash='dash'),
            mode='lines'
        ))
        
        # Forecast line
        fig.add_trace(go.Scatter(
            x=forecast_df['time_index'],
            y=forecast_df['Predicted_Price'],
            name="AI Price Forecast",
            line=dict(color='#ffb703', width=3.5, dash='dash'),
            mode='lines+markers',
            marker=dict(size=8, symbol='diamond')
        ))
        
        fig.update_layout(
            title=f"<b>Weekly Crop Price Trend & AI Forecast - {market_name}</b>",
            xaxis_title="Time Index (Weeks)",
            yaxis_title="Price (Rs per Quintal)",
            template="plotly_dark",
            paper_bgcolor='#0a120f',
            plot_bgcolor='#0b1511',
            font=dict(color='#e0e0e0'),
            margin=dict(l=40, r=40, t=50, b=40),
            hovermode="x unified"
        )
        html_chart = fig.to_html(include_plotlyjs='cdn', full_html=False)
        display(HTML(html_chart))
        
    # Generate Recommendations for the best market within the same District
    with output_recommend:
        clear_output(wait=True)
        district = market_df['District'].iloc[0]
        
        # Find predicted prices for all markets in the same district for week 289
        district_markets = df[df['District'] == district]['Market'].unique()
        district_preds = []
        for dm in district_markets:
            dm_forecast = generate_forecast(dm)
            if not dm_forecast.empty:
                district_preds.append({
                    'Market': dm,
                    'Price': dm_forecast['Predicted_Price'].iloc[0]
                })
        dist_df = pd.DataFrame(district_preds).sort_values('Price', ascending=False)
        
        html_recommend = f"""
        <div class="recommend-card-grid">
            <div class="recommend-card">
                <div class="recommend-header">🏆 District Selling Optimization ({district} District)</div>
                <p>Comparing next week's predicted prices for nearby markets to maximize your sale value:</p>
                <table style='width:100%; border-collapse: collapse; margin-top:10px;'>
                    <tr style='border-bottom: 1px solid #1c4d37; color:#8bada0;'>
                        <th style='text-align:left; padding:8px;'>Market</th>
                        <th style='text-align:right; padding:8px;'>Predicted Price</th>
                        <th style='text-align:right; padding:8px;'>Premium Diff</th>
                    </tr>
        """
        base_val = dist_df[dist_df['Market'] == market_name]['Price'].iloc[0]
        
        for idx, r in dist_df.iterrows():
            diff = r['Price'] - base_val
            diff_str = f"+Rs {diff:.2f}" if diff > 0 else (f"-Rs {abs(diff):.2f}" if diff < 0 else "(Selected)")
            diff_color = "#2ec4b6" if diff > 0 else ("#ff6b6b" if diff < 0 else "#ffffff")
            weight = "bold" if r['Market'] == market_name else "normal"
            
            html_recommend += f"""
                    <tr style='border-bottom: 1px solid #0d2117; font-weight:{weight};'>
                        <td style='padding:8px;'>{r['Market']}</td>
                        <td style='text-align:right; padding:8px; color:#ffffff;'>Rs {r['Price']:.2f}</td>
                        <td style='text-align:right; padding:8px; color:{diff_color}; font-weight:bold;'>{diff_str}</td>
                    </tr>
            """
            
        html_recommend += """
                </table>
            </div>
        </div>
        """
        display(HTML(html_recommend))

# Connect dropdown change event
market_dropdown.observe(update_forecast_tab, names='value')
print("Interactive charts & optimization logic connected!")

In [ ]:
# Cell 8: Render Climate Impact Simulator

def run_simulation(btn=None):
    market_name = market_dropdown.value
    rain_shift = rain_slider.value
    temp_shift = temp_slider.value
    
    # Generate forecasts
    base_f = generate_forecast(market_name)
    sim_f = generate_climate_forecast(market_name, rain_shift, temp_shift)
    
    with output_simulator:
        clear_output(wait=True)
        
        fig = go.Figure()
        
        # Baseline Forecast
        fig.add_trace(go.Scatter(
            x=base_f['time_index'],
            y=base_f['Predicted_Price'],
            name="Baseline Forecast",
            line=dict(color='#2ec4b6', width=3),
            mode='lines+markers'
        ))
        
        # Simulated Forecast
        fig.add_trace(go.Scatter(
            x=sim_f['time_index'],
            y=sim_f['Predicted_Price'],
            name="Simulated Climate Forecast",
            line=dict(color='#ff6b6b', width=3, dash='dot'),
            mode='lines+markers',
            marker=dict(symbol='x', size=8)
        ))
        
        fig.update_layout(
            title=f"<b>Climate Simulation Impact - {market_name}</b>",
            xaxis_title="Time Index (Weeks)",
            yaxis_title="Price (Rs per Quintal)",
            template="plotly_dark",
            paper_bgcolor='#0a120f',
            plot_bgcolor='#0b1511',
            font=dict(color='#e0e0e0'),
            margin=dict(l=40, r=40, t=50, b=40),
            hovermode="x unified"
        )
        html_chart = fig.to_html(include_plotlyjs='cdn', full_html=False)
        display(HTML(html_chart))
        
        # Calculate simulation percentage changes
        avg_base = base_f['Predicted_Price'].mean()
        avg_sim = sim_f['Predicted_Price'].mean()
        pct_diff = ((avg_sim - avg_base) / avg_base) * 100
        
        sign = "increase" if pct_diff >= 0 else "decrease"
        color = "#ff6b6b" if pct_diff < 0 else "#2ec4b6"
        
        sim_notes = f"""
        <div class="recommend-card" style="border-color: #6b2d2d; background: linear-gradient(135deg, #1f1010, #0f0707);">
            <div class="recommend-header" style="color: #ff6b6b;">🌡️ Simulation Analysis Insights</div>
            <p style="font-size: 1.1em; color:#ffd3d3;">
                Under a simulated climate shift of <b>{rain_shift:+.1f}% rainfall</b> and <b>{temp_shift:+.1f}°C temperature</b>, 
                the predicted average price for the next 4 weeks shifts by <span style="color:{color}; font-weight:bold;">{pct_diff:+.2f}%</span> 
                ({sign} from <b>Rs {avg_base:.2f}</b> to <b>Rs {avg_sim:.2f}</b>).
            </p>
            <p style="font-size: 0.9em; color:#d1b6b6; margin-top:5px;">
                *Agricultural Context:* Drought/dry anomalies typically increase temperature stress, reducing yield potentials, leading to a supply squeeze that pushes prices upward. 
                Conversely, cooler anomalies or balanced rains stabilize arrivals and normalize pricing patterns.
            </p>
        </div>
        """
        display(HTML(sim_notes))

def reset_simulation(btn=None):
    rain_slider.value = 0.0
    temp_slider.value = 0.0
    run_simulation()

sim_btn.on_click(run_simulation)
reset_sim_btn.on_click(reset_simulation)

print("Climate Simulator event listeners defined!")

In [ ]:
# Cell 9: Render Alert Center

def update_alerts_tab():
    alerts_df = get_price_crash_alerts(target_week=289)
    
    with output_alerts:
        clear_output(wait=True)
        
        if alerts_df.empty:
            html_content = """
            <div class="recommend-card">
                <div class="recommend-header">✅ Alert Status: Normal</div>
                <p style="color:#a2c9b4;">No major price crash alerts (>10% predicted drops) detected across any of the 80 APMC markets for the next week.</p>
            </div>
            """
            display(HTML(html_content))
            return
            
        html_content = """
        <div>
            <p>The following markets are predicted to suffer a price drop of <b>over 10%</b> next week compared to their last known price. Sellers are advised to complete sales early.</p>
            <div class="alert-card-grid">
        """
        for idx, row in alerts_df.iterrows():
            html_content += f"""
                <div class="alert-card">
                    <div class="alert-header">⚠️ Price Crash Warning</div>
                    <div style="font-size:1.15em; font-weight:bold;">{row['Market']}</div>
                    <div style="color:#bfa3a3; font-size:0.9em; margin-bottom:8px;">District: {row['District']}</div>
                    <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
                        <span>Last Price:</span>
                        <span style="font-weight:bold;">Rs {row['Last_Price']:.2f}</span>
                    </div>
                    <div style="display:flex; justify-content:space-between; margin-bottom:4px;">
                        <span>Predicted Price:</span>
                        <span style="font-weight:bold; color:#ff6b6b;">Rs {row['Predicted_Price']:.2f}</span>
                    </div>
                    <div style="display:flex; justify-content:space-between; margin-top:8px; border-top:1px solid #4a1f1f; padding-top:6px; font-weight:bold;">
                        <span>Expected Crash:</span>
                        <span style="color:#ff6b6b;">-{row['Drop_Pct']:.1f}%</span>
                    </div>
                </div>
            """
        html_content += """
            </div>
        </div>
        """
        display(HTML(html_content))

print("Alert center renderer defined!")

In [ ]:
# Cell 10: Render Farmer Chatbot

chat_history = [
    ("bot", "👋 Namaskar! I am your **AgriSense AI Assistant** 🌾. I can provide price forecasts, identify crash alerts, recommend optimal selling markets, and answer weather impact queries. Try asking me something!\n\n💡 *Tip: Paste a free Gemini API Key below to unlock smart, conversational AI RAG mode!*")
]

chat_log = widgets.Output()
chat_input = widgets.Text(
    placeholder='Type your agricultural query here...',
    layout=widgets.Layout(width='80%')
)
chat_send_btn = widgets.Button(
    description="Send",
    button_style='success',
    icon='paper-plane'
)

gemini_api_key_input = widgets.Password(
    placeholder='Paste your Gemini API Key here (Optional)',
    description='Gemini Key:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='40%')
)

def render_chat_log():
    with chat_log:
        clear_output()
        html_history = "<div class='chat-container'>"
        for speaker, msg in chat_history:
            msg_formatted = msg.replace("**", "<b>").replace("**", "</b>")
            msg_formatted = msg_formatted.replace("*", "<i>").replace("*", "</i>")
            msg_formatted = msg_formatted.replace("\n", "<br/>")
            bubble_class = "chat-bubble-user" if speaker == "user" else "chat-bubble-bot"
            html_history += f"<div class='{bubble_class}'>{msg_formatted}</div>"
        html_history += "</div>"
        display(HTML(html_history))

def handle_rule_based_response(query):
    q = query.lower()
    if any(k in q for k in ['crash', 'alert', 'drop', 'warning', 'fall', 'nuksan']):
        alerts = get_price_crash_alerts(target_week=289)
        if alerts.empty:
            return "✅ **Good News!** No major price crashes (>10% drop) are predicted across any APMC markets for next week. The overall price outlook is stable."
        else:
            response = "⚠️ **Price Crash Warnings for Next Week:**\nBased on next week's AI forecasts, the following markets are showing expected price drops (>10%):\n"
            for i, r in alerts.head(4).iterrows():
                response += f"- **{r['Market']}**: expected drop of **-{r['Drop_Pct']:.1f}%** (Rs {r['Predicted_Price']:.2f} vs Rs {r['Last_Price']:.2f})\n"
            response += "\nWe recommend selling early in these markets or exploring alternative centers nearby."
            return response
    if any(k in q for k in ['recommend', 'best market', 'where to sell', 'highest price', 'sell', 'bazaar', 'profit']):
        best_df = get_best_markets(target_week=289)
        response = "🏆 **Top Recommended Selling Markets Next Week:**\nHere are the top 4 markets with the highest predicted crop prices next week:\n"
        for i, r in best_df.head(4).iterrows():
            response += f"{i+1}. **{r['Market']}** ({r['District']}): **Rs {r['Predicted_Price']:.2f}/quintal**\n"
        response += "\nIf you are near these areas, selling there will maximize your revenues!"
        return response
    if any(k in q for k in ['forecast', 'predict', 'price of', 'trend', 'future', 'bhaav', 'rate']):
        matched_market = None
        for m in df['Market'].unique():
            core_name = m.replace(" APMC", "").lower()
            if core_name in q:
                matched_market = m
                break
        if matched_market:
            f_df = generate_forecast(matched_market)
            response = f"📈 **Price Forecast for {matched_market} (Next 4 Weeks):**\n"
            for idx, row in f_df.iterrows():
                response += f"- Week {idx+1} (time index {int(row['time_index'])}): **Rs {row['Predicted_Price']:.2f}/quintal**\n"
            response += "\nYou can view these results visually on the *Forecast* tab!"
            return response
        else:
            return "Which market's price forecast would you like to see? Please type the market name (e.g., 'pune forecast' or 'price of Ahmednagar APMC'). I support 80 markets!"
    if any(k in q for k in ['weather', 'climate', 'rain', 'drought', 'temp', 'monsoon', 'garmi', 'barish']):
        return "⛈️ **Climate & Weather Impacts:**\nHistorically, temperature changes and rain anomalies affect crop arrivals, which directly drives prices. In our models, cooler periods increase pricing structures, while droughts (low rainfall) generally trigger price rises by 4% to 15% due to crop supply constraints. You can run simulated droughts or heatwaves in the **Climate Simulator** tab!"
    return "👋 Hello! I am your **AgriSense AI Assistant** 🌾. I can help you forecast crop prices, optimize your sale markets, review crash alerts, or understand climate impacts.\n\nTry asking me:\n- *'Will prices crash next week?'*\n- *'Where is the best market to sell crop?'*\n- *'Price of Ahmednagar APMC'*\n- *'How does weather affect crop prices?'*"

def handle_chatbot_response(query):
    api_key = gemini_api_key_input.value.strip()
    if api_key:
        try:
            import google.generativeai as genai
            genai.configure(api_key=api_key)
            context_items = []
            matched_market = None
            q_lower = query.lower()
            for m in df['Market'].unique():
                core_name = m.replace(" APMC", "").lower()
                if core_name in q_lower:
                    matched_market = m
                    break
            if matched_market:
                f_df = generate_forecast(matched_market)
                fc_str = f"Forecast for {matched_market}: " + ", ".join([f"Week {idx+1}: Rs {row['Predicted_Price']:.2f}" for idx, row in f_df.iterrows()])
                context_items.append(fc_str)
            alerts_df = get_price_crash_alerts(target_week=289)
            if alerts_df.empty:
                context_items.append("No price crash warnings next week.")
            else:
                alert_list = []
                for _, r in alerts_df.head(3).iterrows():
                    alert_list.append(f"{r['Market']}: drop of -{r['Drop_Pct']:.1f}% (Rs {r['Predicted_Price_W289']:.2f} vs Rs {r['Last_Price_W288']:.2f})")
                context_items.append("Price crash warnings: " + "; ".join(alert_list))
            best_df = get_best_markets(target_week=289)
            rec_list = []
            for idx, r in best_df.head(3).iterrows():
                rec_list.append(f"{r['Market']}: Rs {r['Predicted_Price']:.2f}")
            context_items.append("Top recommended markets: " + "; ".join(rec_list))
            avg_price = df[df['time_index'] == 288]['Price_Rs_Per_Quintal'].mean()
            context_items.append(f"Average price at Week 288: Rs {avg_price:.2f}")
            context_str = "\n".join(context_items)
            model_gemini = genai.GenerativeModel('gemini-1.5-flash')
            prompt = f"""You are AgriSense AI, a helpful agricultural forecasting assistant.\n
You have access to a machine learning model predicting crop prices and weather impacts.\n
\n
REAL-TIME MODEL PREDICTION CONTEXT:\n
{context_str}\n
\n
USER QUERY:\n
{query}\n
\n
INSTRUCTIONS:\n
1. Answer the user's query accurately using the data context provided.\n
2. Keep your tone supportive, practical, and clear.\n
3. Use markdown bolding and bullet points to make it easy for farmers/traders to read.\n
4. If the query is in Hindi, Hinglish, or Marathi, respond in the corresponding language to be helpful!\n
5. If the query is general agricultural advice not fully covered by the context, combine the context with your general knowledge but stay relevant to market prices, weather, and crops.\n
"""
            response = model_gemini.generate_content(prompt)
            return response.text
        except Exception as e:
            fallback = handle_rule_based_response(query)
            return f"*(Gemini AI Error: {str(e)}. Running in Standard Mode)*\n\n" + fallback
    else:
        return handle_rule_based_response(query)

def send_message(btn=None):
    user_msg = chat_input.value.strip()
    if not user_msg:
        return
    chat_history.append(("user", user_msg))
    chat_input.value = ""
    render_chat_log()
    bot_msg = handle_chatbot_response(user_msg)
    chat_history.append(("bot", bot_msg))
    render_chat_log()

def click_chip(chip_query):
    chat_input.value = chip_query
    send_message()

chat_send_btn.on_click(send_message)
chat_input.on_submit(send_message)

with chat_log:
    render_chat_log()

print("Farmer chatbot assistant loaded!")

In [ ]:
# Cell 11: Assemble and Display the Interactive Tabs Dashboard

# Create Dashboard Header Card HTML
header_html = """
<div class="dashboard-title-card">
    <h1>AgriSense AI: Interactive Intelligence Dashboard</h1>
    <p>Predict prices, simulate climate impacts, locate premium arbitrage markets, and query our smart AI assistant.</p>
</div>
"""

# 1. Setup Forecast Tab Layout
forecast_controls = widgets.HBox([market_dropdown])
forecast_tab_layout = widgets.VBox([
    widgets.HTML("<h3 class='widget-label' style='color:#ffffff; margin-bottom:5px;'>1. Market Price Forecast</h3>"),
    forecast_controls,
    widgets.HBox([output_forecast, output_recommend], layout=widgets.Layout(align_items='stretch'))
])

# 2. Setup Simulator Tab Layout
simulator_controls = widgets.VBox([
    rain_slider,
    temp_slider,
    widgets.HBox([sim_btn, reset_sim_btn], layout=widgets.Layout(gap='10px', margin='10px 0 0 0'))
], layout=widgets.Layout(padding='10px', border='1px solid #1c3d2f', border_radius='8px', background_color='#111e19'))

simulator_tab_layout = widgets.VBox([
    widgets.HTML("<h3 class='widget-label' style='color:#ffffff; margin-bottom:5px;'>2. Climate & Weather Simulator</h3>"),
    widgets.HTML("<p style='color:#8bada0;'>Move the sliders below to simulate a rainfall deficiency/excess and temperature shifts across crop regions. Our ML model will predict the price shifts in real-time.</p>"),
    simulator_controls,
    output_simulator
])

# 3. Setup Alerts Tab Layout
alerts_tab_layout = widgets.VBox([
    widgets.HTML("<h3 class='widget-label' style='color:#ffffff; margin-bottom:5px;'>3. Price Crash Risk Warnings</h3>"),
    output_alerts
])

# 4. Setup Chatbot Tab Layout
# Quick prompt chips helper buttons
chip1 = widgets.Button(description="Will prices crash next week?", layout=widgets.Layout(width='auto'))
chip2 = widgets.Button(description="Where is the best market to sell crop?", layout=widgets.Layout(width='auto'))
chip3 = widgets.Button(description="What is the price forecast for Ahmednagar?", layout=widgets.Layout(width='auto'))
chip4 = widgets.Button(description="How does weather affect crop prices?", layout=widgets.Layout(width='auto'))

chip1.on_click(lambda b: click_chip("Will prices crash next week?"))
chip2.on_click(lambda b: click_chip("Where is the best market to sell crop?"))
chip3.on_click(lambda b: click_chip("What is the price forecast for Ahmednagar?"))
chip4.on_click(lambda b: click_chip("How does weather affect crop prices?"))

chips_container = widgets.HBox([chip1, chip2, chip3, chip4], layout=widgets.Layout(flex_wrap='wrap', gap='5px', margin='0 0 10px 0'))

chatbot_tab_layout = widgets.VBox([
    widgets.HTML("<h3 class='widget-label' style='color:#ffffff; margin-bottom:5px;'>4. AgriSense Conversational AI Assistant</h3>"),
    widgets.HBox([gemini_api_key_input, widgets.HTML("<span style='color:#8bada0; font-size:0.85em; margin:auto 0 auto 10px;'>💡 Paste a free key from <a href='https://aistudio.google.com/' target='_blank' style='color:#2ec4b6; font-weight:bold;'>Google AI Studio</a> to unlock smart AI RAG mode!</span>")]),
    chat_log,
    chips_container,
    widgets.HBox([chat_input, chat_send_btn], layout=widgets.Layout(gap='10px'))
])

# Assemble everything into a Tab Widget
tabs = widgets.Tab()
tabs.children = [forecast_tab_layout, simulator_tab_layout, alerts_tab_layout, chatbot_tab_layout]
tabs.set_title(0, '📈 Market Forecasts')
tabs.set_title(1, '🌡️ Climate Simulator')
tabs.set_title(2, '⚠️ Price Crash Alerts')
tabs.set_title(3, '💬 AI Farmer Chatbot')

# Display header card, dynamic summary, and tabs widget
display(HTML(header_html))
display(output_summary_cards)
display(tabs)

# Initialize tab contents
update_forecast_tab()
run_simulation()
update_alerts_tab()
render_chat_log()
render_summary_cards()
